# Working with remote data, fully lazily

In this demo and exercise, we will showcase an advantage of `dask` beyond arbitrary slicing. We will show how dask can be used to load remote array data lazily, do some simple processing, and store parts of it on disk in a chunked file format called `zarr`.

In [ ]:
from IPython.core.magic import register_cell_magic
from IPython.display import display
import threading, time, tracemalloc

@register_cell_magic
def monitor(_, cell):
    tracemalloc.start()
    start = time.time()
    stop_event = threading.Event()
    handle = display("starting...", display_id=True)

    def report_loop():
        while not stop_event.wait(0.5):  # poll every 0.5s
            current, peak = tracemalloc.get_traced_memory()
            elapsed = time.time() - start
            handle.update(f"[t={elapsed:6.2f}s] current={current/1e6:.1f}MB peak={peak/1e6:.1f}MB")

    t = threading.Thread(target=report_loop, daemon=True)
    t.start()
    try:
        get_ipython().run_cell(cell)
    finally:
        stop_event.set()
        t.join()

    elapsed = time.time() - start
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"\n--- Monitoring report ---")
    print(f"[memory] current: {current/1e6:.1f} MB, peak: {peak/1e6:.1f} MB")
    print(f"[speed] time elapsed: {elapsed:.3f}s")

Our example data this time comes from live two-photon microscopy. In a nutshell, it's a long video (40'000 frames) from a small microscope implanted into the skull of a mouse. The mouse neurons fluoresce when they are active, resulting in brighter pixels in the video. It is a public dataset hosted on the DANDI archive, part of a larger effort called the MIcrONS project. You can read more information about the dataset on the [DANDI archive](https://dandiarchive.org/dandiset/000402/0.230307.2132) and the [related article](http://dx.doi.org/10.1038/s41586-025-08790-w). Our analysis is a simplified version of what one would typically do with such data: its purpose is to demonstrate key concepts.

We can open the remote file using a helper function...


In [ ]:
%%monitor

from pathlib import Path
import importlib
import sys

helper_path = Path.cwd() / "remote_data_helper.py"
spec = importlib.util.spec_from_file_location("utils", helper_path)
assert spec is not None and spec.loader is not None
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)

sys.modules["helper_module"] = helper

import helper_module

print(f"Loaded helper from: {helper_path}")
print(f"Python platform: {sys.platform}")


In [ ]:
%%monitor

from helper_module import open_remote_file


DANDISET_ID = "000402"
ASSET_PATH = "sub-17797/sub-17797_ses-6-scan-4_behavior+image+ophys.nwb"
remote_file, io = open_remote_file(dandiset_id = DANDISET_ID, asset_path = ASSET_PATH)

... and print some metadata about it..

In [ ]:
%%monitor

print("\n--- Session info ---")
print("Session description:", remote_file.session_description)
print("Session start time:", remote_file.session_start_time)
print("Subject:", remote_file.subject)

print("\n--- Acquisition (lazy) ---")
for name, ts in remote_file.acquisition.items():
    print(f"acquisition/{name}: shape={getattr(ts, 'data', None) and ts.data.shape}")

We note that there are 8 acquisitions called `TwoPhotonSeries` in the dataset, which seem to be arrays of shape (40000, 248,440). The fourth will be our example data for now. We can read it into a Python variable:

In [ ]:
%%monitor

two_photon_series = remote_file.acquisition["TwoPhotonSeries4"]


Let's explore this data in more detail.

In [ ]:
%%monitor

from helper_module import print_in_raw_gb


print(type(two_photon_series))
print(type(two_photon_series.data))

print(two_photon_series.data.shape)
print(two_photon_series.data.ndim)


It seems it has a `.data` attribute, which has type `h5py._hl.dataset.Dataset` and it "looks a bit like an array", because it has:
* a shape
* a datatype


In [ ]:

print(two_photon_series.data[0,0,0])
print(two_photon_series.data.chunks)
print_in_raw_gb(two_photon_series.data)



It even seems to have chunks and can be accessed ("sliced") like a `numpy` array! At >8 GB it's slightly too big for laptops with small RAM: we will therefore process it with the `dask` library. We start by reading it into a `dask` array.

In [ ]:

%%monitor
import dask.array as da

two_photon_dask_array = da.from_array(two_photon_series.data, chunks=two_photon_series.data.chunks)


To get an estimate of where we can see active neurons we can take the mean of the first 500 timepoints.

In [ ]:
%%monitor
time_slice = (slice(50, 550))
average = two_photon_dask_array[time_slice].mean(axis=0)
print_in_raw_gb(two_photon_dask_array[time_slice])

Note that despite us reading an 8GB array into memory, slicing it, and calling `.mean` on it, we have barely used any memory. This is because `dask` evaluates _lazily_: it will only load the data into memory once you actually _need_ it. To force `dask` to load and compute the data into an in-memory `numpy` array, we need to call the `.compute` function.

In [ ]:
%%monitor
average_numpy = average.compute()


We can plot the average.

In [ ]:
%%monitor
import matplotlib.pyplot as plt

plt.figure()
plt.imshow(average_numpy)
plt.show()


Loading data over the internet is even slower than loading from local disk. For further analysis, we slightly artificially only care about a single neuron, which is located around pixels 115 to 140 in `y` and 78 to 95 in `x`
To efficiently analyse this neuron across all time slices, we store just the necessary data to local disk into a Zarr file. Conveniently, the data has very small (14x14) spatial chunks, so downloading and writing those from DANDI should be quick!

In [ ]:
%%monitor
# write to local temp store
from pathlib import Path

out_path = Path("./data/twophoton_series.zarr")
single_neuron_slice = (slice(140,150), slice(415,425))
two_photon_dask_array[:, *single_neuron_slice].to_zarr(out_path, overwrite=True)

Finally, we close the file handle for the remote file.

In [ ]:
io.close()

## Stretch exercise

Try combining several `TwoPhotonSeries` lazily into the same dask array, in a new dimension (depth). This will result in a 3D video. Hint: Use `da.stack`.

Alternatively, feel free to try out some of the concepts presented here on your own data, invent your own related stretch exercise or help others in the room.

In [ ]:
%%monitor

import numpy as np
import zarr

two_photon_from_zarr = zarr.open(Path("./data/twophoton_series.zarr"))
print(two_photon_from_zarr.shards)
print(two_photon_from_zarr.chunks)

import math

def n_shard_files(array):
    shape = array.shape
    shards = array.shards
    return math.prod(math.ceil(s / sh) for s, sh in zip(shape, shards))

print(n_shard_files(two_photon_from_zarr))



## Key take-aways

* By keeping our data in `dask` Arrays, we can lazily chain a series of array operations. The chain of operations only gets executed once the data needed is "final", i.e. we call `.compute` or we plot the data.
* When executing the "final" operation, `dask` will load _only_ the required chunks of data into memory.
* With `dask.to_zarr` we can write array data to disk into a "Zarr" file, which under the hood consists of many files (one or a few/chunk) on disk.